In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import  SVC
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier
from pathlib import Path
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline   
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score


PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


X_train_path = PROCESSED_DATA_DIR / "X_train.csv"
y_train_path = PROCESSED_DATA_DIR / "y_train.csv"
X_test_path  = PROCESSED_DATA_DIR / "X_test.csv"

X_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)
X_test  = pd.read_csv(X_test_path)

y_train = y_train.values.ravel()
train_ids = X_train['PassengerId']
test_ids  = X_test['PassengerId']

X_train = X_train.drop(columns=['PassengerId'])
X_test  = X_test.drop(columns=['PassengerId'])


models = [
    ('LR', Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                C=0.01,
                max_iter=1000,
                penalty='l2',
                solver='lbfgs'))
    ])),
    ('RFC', RandomForestClassifier(ccp_alpha=0, max_depth=6, min_samples_split=2, n_estimators=100)),
    ('XGB', XGBClassifier(gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, reg_alpha=0, reg_lambda=1.5))
]

voting_clf = VotingClassifier(
    estimators=models,
    voting='soft',
)

voting_clf.fit(X_train, y_train)

y_pred = voting_clf.predict(X_test).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": y_pred
})

submission.to_csv(OUTPUT_DIR / "comm_predictions.csv", index=False)


print("Predictions saved to:", OUTPUT_DIR / "comm_predictions.csv")
print("=" * 50)
print("BEST MODEL INFORMATION")
print("=" * 50)

scores = cross_val_score(voting_clf, X_train, y_train, cv=5, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")


Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_predictions.csv
BEST MODEL INFORMATION
CV accuracy: 0.7959 ± 0.0127


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import  SVC
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier
from pathlib import Path
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline   
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score


PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


X_train_path = PROCESSED_DATA_DIR / "X_train.csv"
y_train_path = PROCESSED_DATA_DIR / "y_train.csv"
X_test_path  = PROCESSED_DATA_DIR / "X_test.csv"

X_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)
X_test  = pd.read_csv(X_test_path)

y_train = y_train.values.ravel()
train_ids = X_train['PassengerId']
test_ids  = X_test['PassengerId']

X_train = X_train.drop(columns=['PassengerId'])
X_test  = X_test.drop(columns=['PassengerId'])


models = [
    ('LR', Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                C=0.01,
                max_iter=1000,
                penalty='l2',
                solver='lbfgs'))
    ])),
    ('RFC', RandomForestClassifier(ccp_alpha=0, max_depth=6, min_samples_split=2, n_estimators=100)),
    ('XGB', XGBClassifier(gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, reg_alpha=0, reg_lambda=1.5))
]

voting_clf = VotingClassifier(
    estimators=models,
    voting='hard',
)

voting_clf.fit(X_train, y_train)

y_pred = voting_clf.predict(X_test).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": y_pred
})

submission.to_csv(OUTPUT_DIR / "comm_hard_predictions.csv", index=False)


print("Predictions saved to:", OUTPUT_DIR / "comm_hard_predictions.csv")
print("=" * 50)
print("BEST MODEL INFORMATION")
print("=" * 50)

scores = cross_val_score(voting_clf, X_train, y_train, cv=5, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")


Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_hard_predictions.csv
BEST MODEL INFORMATION
CV accuracy: 0.7929 ± 0.0101


# check with different amount of data

In [3]:
feat_imp = pd.read_csv('features_importance.csv')
feat_imp.head()

,Unnamed: 0,feature,importance
0,0,CryoSleep,0.491812
1,9,HomePlanet_Earth,0.095601
2,10,HomePlanet_Europa,0.059875
3,3,RoomService,0.043301
4,6,Spa,0.034521


In [20]:
feat_imp.iloc[:0, 1]

Series([], Name: feature, dtype: object)

In [21]:
PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


X_train_path = PROCESSED_DATA_DIR / "X_train.csv"
y_train_path = PROCESSED_DATA_DIR / "y_train.csv"
X_test_path  = PROCESSED_DATA_DIR / "X_test.csv"

X_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)
X_test  = pd.read_csv(X_test_path)

y_train = y_train.values.ravel()
train_ids = X_train['PassengerId']
test_ids  = X_test['PassengerId']

X_train = X_train.drop(columns=['PassengerId'])
X_test  = X_test.drop(columns=['PassengerId'])

for i in range (20):
    top_features = feat_imp.iloc[:(i+1), 1]
    X_train_i = X_train[top_features]
    X_test_i = X_test[top_features]
    
    models = [
        ('LR', Pipeline([
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(
                    C=0.01,
                    max_iter=1000,
                    penalty='l2',
                    solver='lbfgs'))
        ])),
        ('RFC', RandomForestClassifier(ccp_alpha=0, max_depth=6, min_samples_split=2, n_estimators=100)),
        ('XGB', XGBClassifier(gamma=0, learning_rate=0.1, max_depth=3, n_estimators=100, reg_alpha=0, reg_lambda=1.5))
    ]

    voting_clf = VotingClassifier(
        estimators=models,
        voting='soft',
    )

    voting_clf.fit(X_train_i, y_train)

    y_pred = voting_clf.predict(X_test_i).astype(bool)

    submission = pd.DataFrame({
        "PassengerId": test_ids,
        "Transported": y_pred
    })

    submission.to_csv(OUTPUT_DIR / f"comm_soft_predictions_{i}_features.csv", index=False)


    print("Predictions saved to:", OUTPUT_DIR / f"comm_soft_predictions_{i}_features.csv")
    print("=" * 50)
    print("BEST MODEL INFORMATION")
    print("=" * 50)

    scores = cross_val_score(voting_clf, X_train, y_train, cv=5, scoring="accuracy")
    print(f"CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")


Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_soft_predictions_0_features.csv
BEST MODEL INFORMATION
CV accuracy: 0.7958 ± 0.0167
Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_soft_predictions_1_features.csv
BEST MODEL INFORMATION
CV accuracy: 0.7982 ± 0.0115
Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_soft_predictions_2_features.csv
BEST MODEL INFORMATION
CV accuracy: 0.7981 ± 0.0133
Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_soft_predictions_3_features.csv
BEST MODEL INFORMATION
CV accuracy: 0.7988 ± 0.0128
Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_soft_predictions_4_features.csv
BEST MODEL INFORMATION
CV accuracy: 0.7974 ± 0.0144
Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\comm_soft_predictions_5_features.csv
BEST MOD